# track

> Follow a training wherever it runs: one `SYSONE {json}` progress line per logging step that any log stream carries back, and MLflow runs with parameters, step metrics, system metrics, the training record and the export as an MLflow model.

In [ ]:
#| default_exp track

In [ ]:
#| export
import json, os, shutil, subprocess, sys, tempfile, time, urllib.request
from importlib import metadata
from pathlib import Path

import torch
from fastcore.basics import patch, store_attr
from transformers import TrainerCallback

import sysone
from sysone.learner import Learner
from sysone.record import jsonable, non_default_args, data_record, git_info, entry_point, run_context, lora_config, public_tracking_uri

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne, test_close
import io, logging, contextlib
from transformers import TrainerState
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.inference import Decider
import mlflow
from mlflow import MlflowClient

A training that runs on another machine (Colab, Kaggle, SageMaker) or in the background on this one has to report on itself. Two channels do it here:

- **Progress lines.** `ProgressLog` prints one line per logging step, `SYSONE {"event": "log", "step": 40, "loss": 0.71, ...}`, with the steps per second, the time left and the accelerator's memory and utilisation. Every backend returns a job's standard output somehow (SageMaker's logs, Colab's session, a local file), so a monitor that reads these lines works the same everywhere, with no server to reach.
- **MLflow.** `MLflowLogger` keeps the history: parameters and tags once, the loss and the validation metrics at every step, MLflow's system metrics (CPU, memory, disk, network and NVIDIA GPUs) while training, and, with `log_export`, the training record, the export and an MLflow model that answers like `Decider`. Locally the runs go to a SQLite file in `~/.cache/sysone/mlflow` (MLflow's own file store is in maintenance mode); a server, SageMaker's managed MLflow among them, is one `tracking_uri` away.

```{mermaid}
flowchart LR
    F["learn.fit(...)"] --> P["ProgressLog: SYSONE {json} lines"] --> S["stdout of the job"] --> M["job monitor (cloud)"]
    F --> L["MLflowLogger"] --> DB[("MLflow: SQLite file, local server,<br/>or SageMaker's managed MLflow")]
    E["log_export(learn, path)"] --> DB
    R["a remote job's own SQLite file"] -->|"sync_run, after fetch"| DB
```

The fixtures, with a learner that logs every step:

In [ ]:
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
learn = decision_learner(data, "fixtures/tiny-modernbert", train="lora", preset="cpu", logging_steps=1, disable_tqdm=True)

## Progress lines

`device_stats` reads the memory of the accelerator in use (CUDA's allocated and peak, the MPS allocator's, a TPU's through torch_xla when it is loaded) and the host's CPU and memory use, with `psutil`. GPU utilisation comes from NVIDIA's management library when `nvidia-ml-py` is installed.

In [ ]:
#| export
PREFIX = "SYSONE "

def _gb(b) -> float: return round(b / 2**30, 3)

def device_stats() -> dict:
    "Memory (GB) and utilisation (%) of the accelerator in use and of the host"
    out = {}
    if torch.cuda.is_available():
        out |= {"gpu_mem_gb": _gb(torch.cuda.memory_allocated()), "gpu_peak_gb": _gb(torch.cuda.max_memory_allocated())}
        try: out["gpu_util"] = torch.cuda.utilization()
        except Exception: pass
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        out |= {"mps_mem_gb": _gb(torch.mps.current_allocated_memory()), "mps_driver_gb": _gb(torch.mps.driver_allocated_memory())}
    xm = sys.modules.get("torch_xla.core.xla_model")
    if xm is not None:
        try:
            info = xm.get_memory_info(xm.xla_device())
            used = info.get("bytes_used", info.get("kb_total", 0) * 1024 - info.get("kb_free", 0) * 1024)
            out["xla_mem_gb"] = _gb(used)
        except Exception: pass
    try:
        import psutil
        out |= {"cpu_util": psutil.cpu_percent(None), "ram_pct": psutil.virtual_memory().percent}
    except ImportError: pass
    return out

In [ ]:
st = device_stats()
assert {"cpu_util", "ram_pct"} <= set(st)
st

A line is the prefix and a JSON object; `parse_progress` reads them back out of any text, ignoring everything else a job prints.

In [ ]:
#| export
def _num(v):
    if isinstance(v, str):
        try: return float(v)
        except ValueError: return v
    return v

class ProgressLog(TrainerCallback):
    "Print a `SYSONE {json}` line when a fit starts and ends, at every logging step and at every evaluation"
    def __init__(self, stream=None, hardware:bool=True): self.stream, self.hardware, self._t0 = stream, hardware, None
    def emit(self, **d):
        # in a job, to the process's own stdout: a notebook run by the job captures sys.stdout into its cells
        out = self.stream or (sys.__stdout__ if os.environ.get("SYSONE_JOB") else sys.stdout)
        print(PREFIX + json.dumps(jsonable({"t": round(time.time(), 1)} | d)), file=out, flush=True)
    def on_train_begin(self, args, state, control, **kw):
        if not state.is_world_process_zero: return          # on several GPUs, the main process reports (the others leave _t0 unset)
        self._t0, self._s0 = time.time(), state.global_step
        self.emit(event="start", max_steps=state.max_steps, epochs=args.num_train_epochs, device=args.device.type, **run_context())
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or "loss" not in logs or self._t0 is None: return
        el, done = time.time() - self._t0, state.global_step - self._s0
        sps = done / el if el > 0 and done else None
        self.emit(event="log", step=state.global_step, max_steps=state.max_steps, epoch=round(state.epoch or 0, 3),
                  loss=_num(logs.get("loss")), lr=_num(logs.get("learning_rate")), grad_norm=_num(logs.get("grad_norm")),
                  steps_per_s=round(sps, 3) if sps else None, eta_s=round((state.max_steps - state.global_step) / sps) if sps else None,
                  **(device_stats() if self.hardware else {}))
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if not state.is_world_process_zero: return
        self.emit(event="eval", step=state.global_step, **{k.removeprefix("eval_"): _num(v) for k, v in (metrics or {}).items()})
    def on_train_end(self, args, state, control, **kw):
        if self._t0 is not None: self.emit(event="end", step=state.global_step, seconds=round(time.time() - self._t0, 1))

def parse_progress(text:str) -> list:
    "The progress records in a job's output, in order"
    out = []
    for line in text.splitlines():
        i = line.find(PREFIX + "{")
        if i < 0: continue
        try: out.append(json.loads(line[i + len(PREFIX):]))
        except json.JSONDecodeError: pass
    return out

@patch
def progress(self:Learner, stream=None, hardware:bool=True) -> Learner:
    "Print progress lines from now on (a sysone job turns this on through `SYSONE_PROGRESS`)"
    if not any(isinstance(c, ProgressLog) for c in self.trainer.callback_handler.callbacks): self.trainer.add_callback(ProgressLog(stream, hardware))
    return self

In [ ]:
buf = io.StringIO()
learn.trainer.add_callback(ProgressLog(stream=buf))
learn.fit_one_cycle(1, lr=1e-3)
recs = parse_progress("pip noise\n" + buf.getvalue() + "more noise")
test_eq([r["event"] for r in recs][:2] + [recs[-1]["event"]], ["start", "log", "end"])
logs = [r for r in recs if r["event"] == "log"]
test_eq([r["step"] for r in logs], list(range(1, len(logs) + 1)))
assert all(isinstance(r["loss"], float) for r in logs) and any(r["event"] == "eval" for r in recs)
learn.trainer.remove_callback(ProgressLog)
print(buf.getvalue().splitlines()[1])

On several GPUs (`torchrun`) every process trains and calls the callbacks; as transformers' own progress bar, `ProgressLog` reports from the main process only, so a job's log has one line per step:

In [ ]:
quiet, other = io.StringIO(), TrainerState(is_world_process_zero=False)
pl = ProgressLog(stream=quiet)
for hook, kw in (("on_train_begin", {}), ("on_log", {"logs": {"loss": 1.0}}), ("on_evaluate", {"metrics": {"eval_loss": 1.0}}), ("on_train_end", {})):
    getattr(pl, hook)(None, other, None, **kw)
test_eq(quiet.getvalue(), "")

::: {.callout-note title="Finding: every GPU of a job reported its progress, and wrote the export"}
On Kaggle's two T4s, a job runs its script once per GPU (`torchrun`), and transformers calls the callbacks in every process. Each progress line came twice, and both processes wrote the export to the same folder at once. `ProgressLog` and `MLflowLogger` now act in the main process only, as transformers' own progress bar does, and `learn.export` writes from the main process while the others wait (see [learner](05_learner.ipynb#answers-and-exports)).
:::

## MLflow

`tracking_uri` decides where runs go: the one given, `MLFLOW_TRACKING_URI`, or a SQLite file under `~/.cache/sysone/mlflow` (`SYSONE_MLFLOW_DIR` moves it). A SageMaker MLflow tracking server or app is named by its ARN, and needs the `sagemaker-mlflow` plugin, which signs the requests with your AWS credentials; pin the MLflow client to the server's version (3.0 for tracking servers, 3.10 for MLflow apps as of 2026).

::: {.callout-note title="Finding: MLflow's file store refuses to log, and a SQLite store keeps artifacts under the current folder"}
With MLflow 3.16, a `file:` tracking URI (the `./mlruns` folder of older versions) raises: the file store is in maintenance mode, and runs belong in a database. A SQLite store works, but its experiments put their artifacts in `./mlruns` under whatever folder the process runs in, so models logged from a notebook landed in the notebook's folder. sysone makes each experiment with its artifacts beside the database (`artifact_root`).
:::

In [ ]:
#| export
def mlflow_dir() -> Path: return Path(os.environ.get("SYSONE_MLFLOW_DIR") or Path.home() / ".cache" / "sysone" / "mlflow")

def tracking_uri(uri:str|None=None) -> str:
    "Where MLflow runs go: `uri`, MLFLOW_TRACKING_URI, else a SQLite file in ~/.cache/sysone/mlflow"
    if uri: return uri
    if os.environ.get("MLFLOW_TRACKING_URI"): return os.environ["MLFLOW_TRACKING_URI"]
    mlflow_dir().mkdir(parents=True, exist_ok=True)
    return f"sqlite:///{mlflow_dir() / 'mlflow.db'}"

def artifact_root(uri:str) -> str|None:
    "Where a SQLite store keeps artifacts: a folder beside its database (MLflow's default, ./mlruns, would follow the current folder); None for a server, which decides"
    if not uri.startswith("sqlite:///"): return None
    return (Path(uri[len("sqlite:///"):]).resolve().parent / "artifacts").as_uri()

def use_experiment(name:str, uri:str):
    "Make `name` the active MLflow experiment, creating it with its artifacts beside a SQLite store's database"
    import mlflow
    mlflow.set_tracking_uri(uri)
    if mlflow.get_experiment_by_name(name) is None: mlflow.create_experiment(name, artifact_location=artifact_root(uri))
    return mlflow.set_experiment(name)

def is_sagemaker_mlflow(uri:str) -> bool:
    "Whether a tracking URI names SageMaker's managed MLflow (a tracking server's or an MLflow app's ARN)"
    return uri.startswith("arn:aws:sagemaker:") and (":mlflow-tracking-server/" in uri or ":mlflow-app/" in uri)

def check_tracking_uri(uri:str) -> str:
    "A SageMaker MLflow ARN needs the `sagemaker-mlflow` plugin; anything else MLflow reads itself"
    if is_sagemaker_mlflow(uri):
        try: import sagemaker_mlflow  # noqa: F401
        except ImportError: raise ImportError("a SageMaker MLflow ARN needs the plugin: pip install sagemaker-mlflow") from None
    return uri

In [ ]:
os.environ["SYSONE_MLFLOW_DIR"] = tempfile.mkdtemp()
test_eq(tracking_uri().endswith("mlflow.db") and tracking_uri().startswith("sqlite:///"), True)
arn = "arn:aws:sagemaker:us-east-1:123456789012:mlflow-app/app-abc"
test_eq((is_sagemaker_mlflow(arn), is_sagemaker_mlflow("http://127.0.0.1:5000")), (True, False))
td = tempfile.mkdtemp()
test_eq(artifact_root(f"sqlite:///{td}/mlflow.db"), (Path(td).resolve() / "artifacts").as_uri())

`MLflowLogger` makes one run per learner. The first fit starts it and logs what does not change (the encoder, regime, LoRA settings, head, loss, data, seed and the `TrainingArguments` that differ from transformers' defaults) with the code's commit as MLflow's own `mlflow.source.git.commit` tag; later fits resume it, so a head-then-encoder schedule reads as one curve, its steps counted across the fits. Each fit's summary (from `FitLog`) is logged at its end, and the learner keeps the run's id in `learn.tracking`, which the [training record](05a_record.ipynb) then holds. The range test of `lr_find` is not logged.

In [ ]:
#| export
def learner_params(learn) -> dict:
    "What a learner is, as MLflow parameters: model, data and the non-default training arguments (prefixed `args.`)"
    m, d = learn.model, data_record(learn.data)
    lora, src = lora_config(m) or {}, d.get("source") or {}
    p = {"encoder": m.spec.id if m.spec is not None else None, "regime": m.regime, "head_layers": m.head.layers,
         "head_scorer": m.head.scorer_kind, "loss": getattr(learn.loss, "__name__", type(learn.loss).__name__), "preset": learn.preset,
         "lr": learn.lr, "seed": learn.args.seed, "lora_r": lora.get("r"), "lora_alpha": lora.get("lora_alpha"),
         "lora_targets": lora.get("target_modules"), "data_kind": src.get("kind"), "data_id": src.get("id") or src.get("registry"),
         "template": d.get("template")} | {f"data_{k}": v["cases"] for k, v in d["splits"].items()}
    p |= {f"args.{k}": v for k, v in non_default_args(learn.args).items()}
    return {k: str(jsonable(v))[:6000] for k, v in p.items() if v is not None}

def learner_tags(learn) -> dict:
    "Tags of a run: the code's commit (MLflow's own tag), sysone's version and where the run happens"
    g, entry, run = git_info() or {}, entry_point() or {}, run_context()
    t = {"sysone.version": sysone.__version__, "sysone.backend": run.get("backend"), "sysone.job": run.get("job"),
         "mlflow.source.git.commit": g.get("commit"), "mlflow.source.git.branch": g.get("branch"),
         "sysone.git.dirty": str(g.get("dirty")) if g else None, "mlflow.source.name": entry.get("name")}
    return {k: str(v) for k, v in t.items() if v is not None}

class MLflowLogger(TrainerCallback):
    "Log a learner's fits to one MLflow run: parameters and tags, step metrics, system metrics and fit summaries; later fits resume it"
    def __init__(self, learn, experiment:str|None=None, run_name:str|None=None, uri:str|None=None, system_metrics:bool=True,
                 tags:dict|None=None):
        store_attr(); self.run_id, self.offset, self._on = None, 0, False

    def _begin(self):
        import mlflow
        exp = use_experiment(self.experiment or os.environ.get("MLFLOW_EXPERIMENT_NAME") or "sysone", check_tracking_uri(tracking_uri(self.uri)))
        if mlflow.active_run() is not None and mlflow.active_run().info.run_id != self.run_id: mlflow.end_run()
        if self.run_id is None:
            run = mlflow.start_run(run_name=self.run_name, tags=learner_tags(self.learn) | (self.tags or {}), log_system_metrics=self.system_metrics)
            self.run_id = run.info.run_id
            mlflow.log_params(learner_params(self.learn))
        elif mlflow.active_run() is None: mlflow.start_run(run_id=self.run_id, log_system_metrics=self.system_metrics)
        self.learn.tracking = {"mlflow": {"run_id": self.run_id, "experiment_id": exp.experiment_id,
                                          "tracking_uri": public_tracking_uri(mlflow.get_tracking_uri())}}

    def on_train_begin(self, args, state, control, **kw):
        sch = self.learn.trainer.schedule
        self._on = state.is_world_process_zero and not (sch is not None and sch[0] == "range")    # the main process logs; lr_find's range test is not logged
        if self._on: self._begin()

    def on_log(self, args, state, control, logs=None, **kw):
        if not self._on or not logs: return
        import mlflow
        m = {k: float(_num(logs[k])) for k in ("loss", "learning_rate", "grad_norm") if k in logs and _num(logs[k]) is not None}
        if "loss" in logs: m |= {f"device.{k}": float(v) for k, v in device_stats().items() if k not in ("cpu_util", "ram_pct")}
        if m: mlflow.log_metrics(m, step=self.offset + state.global_step)

    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if not self._on or not metrics: return
        import mlflow
        mlflow.log_metrics({f"valid.{k.removeprefix('eval_')}": float(v) for k, v in metrics.items()
                            if isinstance(_num(v), (int, float)) and not k.endswith(("runtime", "per_second"))}, step=self.offset + state.global_step)

    def on_train_end(self, args, state, control, **kw):
        if not self._on: return
        import mlflow
        fits = self.learn.fit_log.fits
        if fits:
            f, i = fits[-1], len(fits)
            mlflow.log_metrics({k: float(f[k]) for k in ("seconds", "peak_memory_gb", "final_loss") if isinstance(f.get(k), (int, float))} |
                               {"steps": float(f.get("steps") or 0)}, step=i)
            mlflow.set_tag(f"sysone.fit{i}", json.dumps(jsonable(f)))
        self.offset += state.global_step
        mlflow.end_run()
        self._on = False

@patch
def track(self:Learner, experiment:str|None=None, run_name:str|None=None, uri:str|None=None, system_metrics:bool=True, **tags) -> Learner:
    "Log this learner's fits to MLflow (see `MLflowLogger`); a sysone job turns this on through `SYSONE_TRACK`"
    if not any(isinstance(c, MLflowLogger) for c in self.trainer.callback_handler.callbacks):
        self.trainer.add_callback(MLflowLogger(self, experiment, run_name, uri, system_metrics, tags or None))
    return self

In [ ]:
learn2 = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", logging_steps=1, disable_tqdm=True)
mlflow.set_system_metrics_sampling_interval(1); mlflow.set_system_metrics_samples_before_logging(1)
learn2.track(experiment="tiny", run_name="head then lora")
learn2.lr_find(start=1e-5, end=1, steps=10)
learn2.fit(1, lr=1e-3)
learn2.unfreeze()
learn2.fit(1, lr=1e-3)
run = MlflowClient(tracking_uri()).get_run(learn2.tracking["mlflow"]["run_id"])
test_eq((run.data.params["regime"], run.data.params["encoder"], run.data.params["data_train"]), ("head", "fixtures/tiny-modernbert", "12"))
test_eq(run.data.tags["mlflow.source.git.commit"], git_info()["commit"])
hist = MlflowClient(tracking_uri()).get_metric_history(run.info.run_id, "loss")
test_eq(sorted(m.step for m in hist), list(range(1, len(hist) + 1)))      # two fits, one curve, the range test left out
assert {"sysone.fit1", "sysone.fit2"} <= set(run.data.tags) and "valid.accuracy" in run.data.metrics
sorted(run.data.metrics)

Inside a job, the learner turns both on by itself: `SYSONE_PROGRESS` adds the progress lines, and `SYSONE_TRACK` (`1`, or a JSON object of `learn.track` options) the MLflow run. The [jobs](40_cloud.ipynb) set them, so a training script needs no change to report from Colab, Kaggle or SageMaker.

In [ ]:
os.environ |= {"SYSONE_PROGRESS": "1", "SYSONE_TRACK": json.dumps({"experiment": "a job", "system_metrics": False})}
lj = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", disable_tqdm=True)
test_eq(sorted(type(c).__name__ for c in lj.trainer.callback_handler.callbacks if type(c).__name__ in ("ProgressLog", "MLflowLogger")),
        ["MLflowLogger", "ProgressLog"])
test_eq(next(c for c in lj.trainer.callback_handler.callbacks if type(c).__name__ == "MLflowLogger").experiment, "a job")
del os.environ["SYSONE_PROGRESS"], os.environ["SYSONE_TRACK"]

## The export as an MLflow model

`log_export` adds a training's result to its run: `sysone.json` and `environment.txt` as artifacts, and an MLflow model that answers like `Decider`. The model is logged as code, not pickled: a small module defines `SysoneModel`, whose context loads the export, and sysone's own source travels with it, so the model loads wherever its requirements install, sysone on PyPI or not. Its input is a list of records with a `state` and `questions`, its output their answers; MLflow reads the signature from the type hints. `register="name"` also puts it in MLflow's model registry.

In [ ]:
#| export
PYFUNC = '''\
# A sysone export as an MLflow model, logged by sysone.track.log_export
from typing import Any
import mlflow
from mlflow.pyfunc import PythonModel

class SysoneModel(PythonModel):
    "Answers in the Jev schema for records with a state and questions"
    def load_context(self, context):
        from sysone.inference import Decider
        self.decider = Decider.load(context.artifacts["export"], device="cpu")
    def predict(self, model_input: list[dict[str, Any]], params=None) -> list[dict[str, Any]]:
        return [self.decider.predict(r["state"], r["questions"]) for r in model_input]

mlflow.models.set_model(SysoneModel())
'''

def model_requirements() -> list:
    "What an MLflow model of an export installs: sysone's own requirements (its source travels with the model)"
    reqs = [r for r in (metadata.requires("sysone") or []) if "extra ==" not in r]
    return reqs or ["torch", "transformers", "peft", "safetensors", "huggingface_hub", "datasets", "numpy", "fastcore", "plum-dispatch"]

def log_export(learn, path=None, register:str|None=None, example:dict|None=None, uri:str|None=None) -> str:
    "Log an export (made from `learn` if no `path`) to the learner's MLflow run: files, and an MLflow model answering like Decider"
    import mlflow
    if not getattr(learn, "tracking", None): raise ValueError("the learner has no MLflow run: call learn.track() before fitting")
    path = Path(path) if path is not None else learn.export(Path(tempfile.mkdtemp()) / "export")
    mlflow.set_tracking_uri(check_tracking_uri(tracking_uri(uri)))
    code = Path(tempfile.mkdtemp()) / "sysone_model.py"; code.write_text(PYFUNC)
    with mlflow.start_run(run_id=learn.tracking["mlflow"]["run_id"]):
        for f in ("sysone.json", "environment.txt"):
            if (path / f).exists(): mlflow.log_artifact(str(path / f), artifact_path="export")
        info = mlflow.pyfunc.log_model(name="model", python_model=str(code), artifacts={"export": str(path)},
                                       code_paths=[str(Path(sysone.__file__).parent)], pip_requirements=model_requirements(),
                                       input_example=[example] if example else None, registered_model_name=register)
    return info.model_uri

In [ ]:
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
ex = {"state": rec["state"], "questions": rec["questions"]}
model_uri = log_export(learn2, example=ex, register="tiny-jev")
loaded = mlflow.pyfunc.load_model(model_uri)
test_eq(loaded.predict([ex]), [learn2.decider(device="cpu").predict(rec["state"], rec["questions"])])
test_eq(MlflowClient(tracking_uri()).get_registered_model("tiny-jev").name, "tiny-jev")
arts = [a.path for a in MlflowClient(tracking_uri()).list_artifacts(learn2.tracking["mlflow"]["run_id"], "export")]
test_eq(sorted(arts), ["export/environment.txt", "export/sysone.json"])

The export carries the run too: its training record names the run that logged it.

In [ ]:
meta = json.loads((learn2.export(Path(tempfile.mkdtemp()) / "tracked") / "sysone.json").read_text())
test_eq(meta["training"]["tracking"]["mlflow"]["run_id"], learn2.tracking["mlflow"]["run_id"])
test_eq(meta["training"]["tracking"]["mlflow"]["tracking_uri"], "sqlite")

## Runs logged elsewhere

A job on Colab or Kaggle cannot reach a server on your laptop, so it logs to a SQLite file of its own, which comes back with the job's other outputs. `sync_run` copies its runs into your store: parameters, tags, every point of every metric, and the artifacts (models logged with `log_export` are re-logged from the export by the job's fetch).

In [ ]:
#| export
def sync_run(src:str, dest:str|None=None, run_id:str|None=None, experiment:str|None=None) -> list:
    "Copy runs (all of `src`'s, or `run_id`) with their parameters, tags, metric histories and artifacts into `dest`; returns the new run ids"
    from mlflow import MlflowClient
    from mlflow.entities import Metric, Param
    s, d = MlflowClient(src), MlflowClient(tracking_uri(dest))
    runs = [s.get_run(run_id)] if run_id else [r for e in s.search_experiments() for r in s.search_runs([e.experiment_id])]
    out = []
    for r in runs:
        name = experiment or s.get_experiment(r.info.experiment_id).name
        exp = d.get_experiment_by_name(name)
        eid = exp.experiment_id if exp else d.create_experiment(name, artifact_location=artifact_root(tracking_uri(dest)))
        tags = {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.log-model")} | {"sysone.synced_from": r.info.run_id}
        new = d.create_run(eid, start_time=r.info.start_time, tags=tags, run_name=r.info.run_name)
        metrics = [Metric(m.key, m.value, m.timestamp, m.step) for k in r.data.metrics for m in s.get_metric_history(r.info.run_id, k)]
        params = [Param(k, v) for k, v in r.data.params.items()]
        for i in range(0, len(params), 100): d.log_batch(new.info.run_id, params=params[i:i + 100])     # MLflow's batch limits
        for i in range(0, len(metrics), 1000): d.log_batch(new.info.run_id, metrics=metrics[i:i + 1000])
        tmp = Path(tempfile.mkdtemp())
        if s.list_artifacts(r.info.run_id):
            local = s.download_artifacts(r.info.run_id, "", str(tmp))
            d.log_artifacts(new.info.run_id, local)
        d.set_terminated(new.info.run_id, r.info.status, end_time=r.info.end_time)
        out.append(new.info.run_id)
    return out

In [ ]:
remote = f"sqlite:///{tempfile.mkdtemp()}/job.db"
learn3 = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", logging_steps=1, disable_tqdm=True)
learn3.track(experiment="on colab", uri=remote, system_metrics=False)
learn3.fit(1, lr=1e-3)
new = sync_run(remote)
nr = MlflowClient(tracking_uri()).get_run(new[0])
test_eq((nr.data.tags["sysone.synced_from"], nr.data.params["regime"]), (learn3.tracking["mlflow"]["run_id"], "head"))
test_eq(len(MlflowClient(tracking_uri()).get_metric_history(new[0], "loss")), len(MlflowClient(remote).get_metric_history(learn3.tracking["mlflow"]["run_id"], "loss")))

## A local server

`mlflow_server()` starts MLflow's server over the same SQLite file in the background (the UI at its address), and `.stop()` ends it; `MLFLOW_TRACKING_URI` then points every run of the session at it.

In [ ]:
#| export
class MLflowServer:
    "A local MLflow server process over a SQLite store"
    def __init__(self, proc, url:str, store:Path): self.proc, self.url, self.store = proc, url, store
    def __repr__(self): return f"MLflowServer({self.url}, store {self.store}, {'running' if self.proc.poll() is None else 'stopped'})"
    def stop(self):
        "Stop the server and the workers it started"
        if self.proc.poll() is None:
            self.proc.terminate()
            try: self.proc.wait(timeout=30)
            except subprocess.TimeoutExpired: self.proc.kill()
        if os.environ.get("MLFLOW_TRACKING_URI") == self.url: del os.environ["MLFLOW_TRACKING_URI"]

def _up(url:str) -> bool:
    try: return urllib.request.urlopen(url + "/health", timeout=2).status == 200
    except Exception: return False

def mlflow_server(port:int=5000, store=None, host:str="127.0.0.1", timeout:float=60, use:bool=True) -> MLflowServer:
    "Start MLflow's server in the background over a SQLite store (~/.cache/sysone/mlflow by default); `use` points this session at it"
    store = Path(store or mlflow_dir()); store.mkdir(parents=True, exist_ok=True)
    url = f"http://{host}:{port}"
    if _up(url): raise RuntimeError(f"something already answers at {url}; pick another port")
    log = open(store / f"server-{port}.log", "ab")
    proc = subprocess.Popen([sys.executable, "-m", "mlflow", "server", "--host", host, "--port", str(port),
                             "--backend-store-uri", f"sqlite:///{store / 'mlflow.db'}", "--artifacts-destination", str(store / "artifacts")],
                            stdout=log, stderr=subprocess.STDOUT, env=os.environ | {"MLFLOW_DISABLE_AGENT_HINT": "1"}, start_new_session=True)
    t0 = time.time()
    while not _up(url):
        if proc.poll() is not None or time.time() - t0 > timeout:
            proc.kill(); raise RuntimeError(f"the MLflow server did not start; see {store / f'server-{port}.log'}")
        time.sleep(0.5)
    if use: os.environ["MLFLOW_TRACKING_URI"] = url
    return MLflowServer(proc, url, store)

In [ ]:
#| mlflow
server = mlflow_server(port=5077, store=tempfile.mkdtemp())
learn4 = decision_learner(data, "fixtures/tiny-modernbert", train="head", preset="cpu", logging_steps=1, disable_tqdm=True).track(experiment="served")
learn4.fit(1, lr=1e-3)
test_eq(MlflowClient(server.url).get_run(learn4.tracking["mlflow"]["run_id"]).data.params["regime"], "head")
test_eq(learn4.tracking["mlflow"]["tracking_uri"], server.url)
server.stop()
server

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()